# Supervised Learning Basics
## Machine learning for energy applications — Module 1, Exercise 1

In this exercise we explore the foundational supervised learning algorithms that appear
throughout modern machine-learning workflows in the energy industry. We will work through
linear regression, regularization, K-nearest neighbours, decision trees, and random forests
using well-known reference datasets from scikit-learn.

**Evaluation habits used throughout.** We split the data before fitting anything. Models and
preprocessing steps are fitted on **training** rows only. Settings are chosen on **validation**
rows (or by cross-validation on the training rows), and every score table starts with a
**baseline**. The **test** rows are opened once, at the end, for the locked choice.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, get_data_home, load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
)

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Linear Regression

We use the **California Housing** dataset, which contains median house values for
California districts derived from the 1990 census. Each sample has eight numerical
features (median income, house age, average rooms, etc.). The target is the median
house value in units of 100,000 dollars. A random split of the rows asks how well the
model predicts other districts from the same regions, which is the question for this
exercise; a model meant for a new region would need a spatially grouped split.

> **Data download.** `fetch_california_housing` needs internet access the first time it runs on a
> machine: it downloads the data (about 0.4 MB) and caches it in the scikit-learn data folder
> (`~/scikit_learn_data` by default, or the folder named by the `SCIKIT_LEARN_DATA` environment
> variable). Offline machines need a pre-filled cache: copy that folder, which then holds
> `cal_housing_py3.pkz`, from a machine where this cell has run. Without internet access or a
> cache, the loading cell stops with an error that says so.

### 1.1 Load Data and Fit a Linear Model

In [ ]:
# Load the California Housing dataset. The first call downloads it and caches it in the
# scikit-learn data folder; later calls read the cached copy (see the note above).
try:
    housing = fetch_california_housing(as_frame=True)
except OSError as err:  # URLError, HTTPError and timeouts are all OSErrors
    raise RuntimeError(
        "California housing is not cached on this machine and could not be downloaded. "
        "Connect to the internet and rerun this cell, or copy a pre-filled scikit-learn "
        f"data folder (containing cal_housing_py3.pkz) to {get_data_home()}"
    ) from err
X_housing = housing.data
y_housing = housing.target  # median house value in $100k

print(f"Features : {list(X_housing.columns)}")
print(f"Samples  : {X_housing.shape[0]}")
print(f"Number of features : {X_housing.shape[1]}")
X_housing.head()

In [ ]:
# Split first, before fitting anything. Hold out 20 % of the rows, then halve them:
#   validation rows (10 %) compare models and choose settings in sections 1-2;
#   test rows (10 %) stay closed until section 2.3, where they are used once.
# 2,064 rows each is plenty for a regression score on a dataset of this size.
X_train_h, X_hold_h, y_train_h, y_hold_h = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=RANDOM_STATE
)
X_val_h, X_test_h, y_val_h, y_test_h = train_test_split(
    X_hold_h, y_hold_h, test_size=0.5, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train_h)}  validation rows: {len(X_val_h)}  "
      f"test rows (closed): {len(X_test_h)}")


def score_row(name, model, X_tr, X_va):
    """Training and validation scores of a model fitted on the training rows."""
    y_va_pred = model.predict(X_va)
    return {
        "model": name,
        "training R²": r2_score(y_train_h, model.predict(X_tr)),
        "validation R²": r2_score(y_val_h, y_va_pred),
        "validation MAE": mean_absolute_error(y_val_h, y_va_pred),
    }


scores = []  # the score table; the baseline row comes first

# Baseline: always predict the mean of the training targets
baseline = DummyRegressor().fit(X_train_h, y_train_h)
scores.append(score_row("Mean baseline (DummyRegressor)", baseline, X_train_h, X_val_h))

# Fit ordinary least-squares linear regression on the eight features
lr = LinearRegression()
lr.fit(X_train_h, y_train_h)
scores.append(score_row("Linear, 8 features", lr, X_train_h, X_val_h))

pd.DataFrame(scores).round(4)

### 1.2 Polynomial Features

Adding polynomial features (degree 2) allows the linear model to capture non-linear
relationships. We scale the data first to keep polynomial terms numerically stable.
The scaler learns means and standard deviations, so it is fitted on the training rows
only; validation rows are only transformed with those training statistics.

In [ ]:
# Scale features before polynomial expansion: fitted on training rows only
scaler = StandardScaler().fit(X_train_h)
poly2 = PolynomialFeatures(degree=2, include_bias=False).fit(scaler.transform(X_train_h))
X_train_poly2 = poly2.transform(scaler.transform(X_train_h))
X_val_poly2 = poly2.transform(scaler.transform(X_val_h))

lr_poly2 = LinearRegression()
lr_poly2.fit(X_train_poly2, y_train_h)

print(f"Polynomial degree 2 — number of features: {X_train_poly2.shape[1]}")
scores.append(score_row("Degree 2, 44 terms", lr_poly2, X_train_poly2, X_val_poly2))
pd.DataFrame(scores).round(4)

### 1.3 Demonstrating Overfitting with High-Degree Polynomials

We use a *single feature* (MedInc) so we can visualize the effect of increasing
polynomial degree, and compare training and validation R² for degrees 1, 2, 5 and 15.
We fit each degree twice: with raw MedInc powers, and with the polynomial columns
standardized, `make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression())`.

A bigger polynomial contains every smaller one, so **training** R² can never fall as the
degree grows. If it does, the least-squares solver has failed on badly scaled columns
(raw MedInc¹⁵ reaches about 15¹⁵ ≈ 4 × 10¹⁷), not the model. How badly it fails depends
on the scikit-learn version; with the columns standardized the fit is stable. With
16,512 training rows and one feature there is little room to overfit at all.

In [ ]:
# Use a single feature for visualization
feature = "MedInc"
X_single_train = X_train_h[[feature]]
X_single_val = X_val_h[[feature]]
x_grid = pd.DataFrame({feature: np.linspace(X_single_train[feature].min(),
                                            X_single_train[feature].max(), 300)})

degrees = [1, 2, 5, 15]
variants = {
    "Raw MedInc powers": lambda d: make_pipeline(
        PolynomialFeatures(degree=d, include_bias=False), LinearRegression()),
    "Standardized polynomial columns": lambda d: make_pipeline(
        PolynomialFeatures(degree=d, include_bias=False), StandardScaler(), LinearRegression()),
}

degree_scores = []
fig, axes = plt.subplots(len(variants), len(degrees), figsize=(20, 8), sharey=True)
for row_axes, (variant, make_model) in zip(axes, variants.items()):
    for ax, d in zip(row_axes, degrees):
        model = make_model(d).fit(X_single_train, y_train_h)  # training rows only
        train_r2 = r2_score(y_train_h, model.predict(X_single_train))
        val_r2 = r2_score(y_val_h, model.predict(X_single_val))
        degree_scores.append({"variant": variant, "degree": d,
                              "training R²": train_r2, "validation R²": val_r2})

        ax.scatter(X_single_val[feature], y_val_h, alpha=0.15, s=5, color="steelblue")
        ax.plot(x_grid[feature], model.predict(x_grid), color="tomato", linewidth=2)
        ax.set_title(f"{variant}, degree {d}\nTraining R²={train_r2:.2f}  "
                     f"Validation R²={val_r2:.2f}", fontsize=10)
        ax.set_xlabel(feature)
    row_axes[0].set_ylabel("Median House Value ($100k)")

fig.suptitle("Polynomial Regression — Training vs Validation R² (markers: validation rows)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(pd.DataFrame(degree_scores).round(3).to_string(index=False))

---
## 2 — Regularization: Ridge (L2) and Lasso (L1)

Regularization adds a penalty to the loss function to prevent large coefficients,
reducing overfitting.

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge (L2)** | Sum of squared coefficients | Shrinks coefficients toward zero |
| **Lasso (L1)** | Sum of absolute coefficients | Drives some coefficients exactly to zero (feature selection) |

We compare α values on the **validation** rows; section 2.3 locks one choice and opens the
test rows once.

### 2.1 Ridge vs Lasso on Polynomial Features

In [ ]:
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

results = []
fitted = {}  # (model name, alpha) -> model fitted on the training rows
for alpha in alphas:
    ridge = Ridge(alpha=alpha).fit(X_train_poly2, y_train_h)
    lasso = Lasso(alpha=alpha, max_iter=10000).fit(X_train_poly2, y_train_h)
    fitted[("Ridge", alpha)], fitted[("Lasso", alpha)] = ridge, lasso

    results.append({
        "alpha": alpha,
        "Ridge validation R²": r2_score(y_val_h, ridge.predict(X_val_poly2)),
        "Lasso validation R²": r2_score(y_val_h, lasso.predict(X_val_poly2)),
        "Ridge non-zero coefs": np.sum(np.abs(ridge.coef_) > 1e-6),
        "Lasso non-zero coefs": np.sum(np.abs(lasso.coef_) > 1e-6),
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

### 2.2 Coefficient Comparison — Lasso Feature Selection

In [ ]:
# Fit with a moderate alpha to highlight the difference
alpha_demo = 0.1
ridge_demo = Ridge(alpha=alpha_demo).fit(X_train_poly2, y_train_h)
lasso_demo = Lasso(alpha=alpha_demo, max_iter=10000).fit(X_train_poly2, y_train_h)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(ridge_demo.coef_)), ridge_demo.coef_, color="steelblue", width=1.0)
axes[0].set_title(f"Ridge (alpha={alpha_demo}) — {np.sum(np.abs(ridge_demo.coef_) > 1e-6)} non-zero coefficients")
axes[0].set_xlabel("Feature index")
axes[0].set_ylabel("Coefficient value")

axes[1].bar(range(len(lasso_demo.coef_)), lasso_demo.coef_, color="tomato", width=1.0)
axes[1].set_title(f"Lasso (alpha={alpha_demo}) — {np.sum(np.abs(lasso_demo.coef_) > 1e-6)} non-zero coefficients")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("Coefficient value")

plt.suptitle("Regularization: Ridge vs Lasso Coefficient Magnitudes", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Lasso zeroed out {np.sum(np.abs(lasso_demo.coef_) <= 1e-6)} of {len(lasso_demo.coef_)} coefficients.")

### 2.3 Lock the Choice, Then Open the Test Rows Once

We choose on validation rows only: the model and α with the highest validation R² in the
table above. Differences of a few thousandths are within the noise of a single validation
split, so preferring the more regularized α within such a plateau is also defensible; what
matters is that the choice is made before the test rows are used. Because we chose on the validation rows,
their best score is optimistic. The test rows have not been used for anything yet: we
score the locked model on them once, next to the baseline on the same rows. Changing the
model after seeing this score would turn the test rows into validation rows.

In [ ]:
# 1. Lock the choice using validation R² only
val_r2 = {key: r2_score(y_val_h, model.predict(X_val_poly2)) for key, model in fitted.items()}
best_name, best_alpha = max(val_r2, key=val_r2.get)
final_model = fitted[(best_name, best_alpha)]
print(f"Locked choice: degree-2 features + {best_name}(alpha={best_alpha}), "
      f"validation R² = {val_r2[(best_name, best_alpha)]:.4f}")

# 2. Open the test rows once: transform them with the scaler and polynomial step
#    fitted on the training rows, then score the locked model and the baseline.
X_test_poly2 = poly2.transform(scaler.transform(X_test_h))
y_test_pred = final_model.predict(X_test_poly2)
y_test_base = baseline.predict(X_test_h)
final_table = pd.DataFrame([
    {"model": "Mean baseline (DummyRegressor)",
     "test R²": r2_score(y_test_h, y_test_base),
     "test MAE": mean_absolute_error(y_test_h, y_test_base)},
    {"model": f"{best_name}(alpha={best_alpha}), degree 2",
     "test R²": r2_score(y_test_h, y_test_pred),
     "test MAE": mean_absolute_error(y_test_h, y_test_pred)},
])
print(f"FINAL TEST ({len(X_test_h)} rows, used once, after the choice was locked):")
print(final_table.round(4).to_string(index=False))

---
## 3 — K-Nearest Neighbours (KNN)

KNN is a simple, instance-based algorithm: to classify a new point, find the *K* closest
training points and take a majority vote. We use the **Iris** dataset (150 samples,
4 features, 3 classes).

Iris is small: after holding out 30 test rows, 120 training rows remain, and one row of a
30-row set is worth 3.3 accuracy points. A single validation split would be too noisy to
compare settings, so we compare them with **5-fold stratified cross-validation on the
training rows** (the same folds for every model) and keep the test rows closed until
section 4.3.

### 3.1 Load Iris Data and Train KNN (K=5)

In [ ]:
iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris = iris.target
print("Label encoding:", dict(enumerate(iris.target_names.tolist())))

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train_i)}  test rows (closed until 4.3): {len(X_test_i)}")

# The same stratified folds for every comparison in sections 3-5
cv_iris = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# All four features are lengths in cm, so raw distances are meaningful here; with
# mixed units, put a StandardScaler inside the pipeline before KNN.
baseline_clf = DummyClassifier(strategy="most_frequent")
baseline_cv = cross_val_score(baseline_clf, X_train_i, y_train_i, cv=cv_iris).mean()

knn = KNeighborsClassifier(n_neighbors=5)
knn_cv = cross_val_score(knn, X_train_i, y_train_i, cv=cv_iris)

print(f"Majority-class baseline CV accuracy: {baseline_cv:.4f}")
print(f"KNN (K=5) CV accuracy: {knn_cv.mean():.4f} ± {knn_cv.std():.4f} "
      f"(5 folds of {len(X_train_i) // 5} training rows)")

### 3.2 Accuracy vs K

In [ ]:
k_values = [1, 3, 5, 10, 20]
train_acc = []
cv_mean = []
cv_std = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    cv_acc = cross_val_score(model, X_train_i, y_train_i, cv=cv_iris)
    cv_mean.append(cv_acc.mean())
    cv_std.append(cv_acc.std())
    model.fit(X_train_i, y_train_i)
    train_acc.append(accuracy_score(y_train_i, model.predict(X_train_i)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, train_acc, "o-", label="Training accuracy", color="steelblue")
ax.errorbar(k_values, cv_mean, yerr=cv_std, fmt="s--", capsize=4,
            label="5-fold CV accuracy (mean ± SD)", color="tomato")
ax.axhline(baseline_cv, color="grey", linestyle=":", label="Majority-class baseline (CV)")
ax.set_xlabel("K (number of neighbours)")
ax.set_ylabel("Accuracy")
ax.set_title("KNN — Accuracy vs K")
ax.set_xticks(k_values)
ax.legend()
plt.tight_layout()
plt.show()

for k, tr, m, s in zip(k_values, train_acc, cv_mean, cv_std):
    print(f"K={k:>2d}  Train={tr:.4f}  CV={m:.4f} ± {s:.4f}")

---
## 4 — Decision Tree

Decision trees recursively partition the feature space using axis-aligned splits.
They are easy to interpret but prone to overfitting if not constrained.

### 4.1 Train and Visualize

In [ ]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_cv = cross_val_score(dt, X_train_i, y_train_i, cv=cv_iris)
dt.fit(X_train_i, y_train_i)

print(f"Decision Tree (no depth limit) CV accuracy: {dt_cv.mean():.4f} ± {dt_cv.std():.4f}")
print(f"Tree depth: {dt.get_depth()}")

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=iris.target_names.tolist(),
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=8,
)
ax.set_title("Decision Tree — Full (No Depth Limit)", fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 Effect of `max_depth`

In [ ]:
depths = [1, 2, 3, 5, None]
depth_cv = {}
fig, axes = plt.subplots(1, len(depths), figsize=(24, 5))

for ax, d in zip(axes, depths):
    tree = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    depth_cv[d] = cross_val_score(tree, X_train_i, y_train_i, cv=cv_iris).mean()
    tree.fit(X_train_i, y_train_i)

    plot_tree(
        tree,
        feature_names=iris.feature_names,
        class_names=iris.target_names.tolist(),
        filled=True,
        rounded=True,
        ax=ax,
        fontsize=7,
    )
    label = "None" if d is None else str(d)
    ax.set_title(f"max_depth={label}\nCV accuracy={depth_cv[d]:.2f}", fontsize=10)

plt.suptitle("Decision Tree — Effect of max_depth", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

for d, m in depth_cv.items():
    print(f"max_depth={str(d):>4s}  CV={m:.4f}")

### 4.3 Lock One Classifier, Then Open the Test Rows Once

The candidates are the K values of section 3.2 and the depths of section 4.2, compared on
the same cross-validation folds. We lock the one with the highest mean CV accuracy (the
first listed wins a tie), fit it on all training rows, and score the 30 test rows once.
With so few rows, the test score can confirm a choice but could not have ranked these
candidates.

In [ ]:
# Candidates compared on the same CV folds (sections 3.2 and 4.2)
candidates = {f"KNN, K={k}": (KNeighborsClassifier(n_neighbors=k), m)
              for k, m in zip(k_values, cv_mean)}
candidates.update({f"Tree, max_depth={d}": (DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE), m)
                   for d, m in depth_cv.items()})

# 1. Lock the choice using CV accuracy on the training rows only
locked_name = max(candidates, key=lambda name: candidates[name][1])
locked_clf, locked_cv = candidates[locked_name]
print(f"Locked choice: {locked_name} (CV accuracy {locked_cv:.4f})")

# 2. Fit it on all training rows and open the test rows once
locked_clf.fit(X_train_i, y_train_i)
n_correct = int((locked_clf.predict(X_test_i) == y_test_i).sum())
baseline_test = accuracy_score(y_test_i, baseline_clf.fit(X_train_i, y_train_i).predict(X_test_i))
print(f"FINAL TEST ({len(y_test_i)} rows, used once): {n_correct}/{len(y_test_i)} correct, "
      f"accuracy {n_correct / len(y_test_i):.4f}")
print(f"Majority-class baseline on the same test rows: accuracy {baseline_test:.4f}")
print(f"One test row is worth {100 / len(y_test_i):.1f} accuracy points.")

---
## 5 — Random Forest

A **Random Forest** is an ensemble of decision trees trained on bootstrapped samples
with random feature subsets. This reduces variance and typically outperforms a single
tree.

The test rows were used once, for the locked choice in section 4.3, so the forest is
compared with the other models by cross-validation on the same training folds only.
(Had it been a candidate, it would have joined the section 4.3 comparison before the test.)

### 5.1 Train and Evaluate

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE
)
rf_cv = cross_val_score(rf, X_train_i, y_train_i, cv=cv_iris)
rf.fit(X_train_i, y_train_i)  # all training rows, for the importances below

print(f"Random Forest CV accuracy: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

### 5.2 Feature Importance

Impurity-based importances describe how this forest uses the features on the training
rows; they are not causal effects. Module 2 compares them with permutation importance.

In [ ]:
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    np.array(iris.feature_names)[sorted_idx],
    importances[sorted_idx],
    color="steelblue",
)
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Random Forest — Feature Importance (Iris Dataset)")
plt.tight_layout()
plt.show()

for name, imp in zip(np.array(iris.feature_names)[sorted_idx], importances[sorted_idx]):
    print(f"  {name:<20s} {imp:.4f}")

---
## Summary

In this exercise we covered:

1. **Linear Regression** — fitting, evaluation metrics (R², MAE), polynomial features, and overfitting, with a mean baseline first in the score table.
2. **Regularization** — Ridge (L2) and Lasso (L1), with Lasso performing automatic feature selection; α chosen on validation rows and the locked model tested once.
3. **K-Nearest Neighbours** — instance-based classification and the effect of K on bias-variance trade-off, compared by cross-validation against a majority-class baseline.
4. **Decision Trees** — interpretable models and the role of `max_depth` in controlling complexity; one locked classifier scored once on the test rows.
5. **Random Forest** — ensemble learning and feature importance.

These algorithms form the building blocks for more advanced techniques used across
the energy industry, from production forecasting to well-log analysis. In those settings,
the split must also follow the deployment question: whole wells or later time periods,
not random rows.